# Thesis Workflow: Dense Baseline vs. HASHI

**Project**: Deep-Learning-based Upsampling of Classification Heatmaps for WSI

## Overview
This notebook recreates the complete experimental pipeline described in the thesis.

**Part 1: The Standard Pipeline (Baseline)**
1.  **Load WSI**: Read the proprietary slide format.
2.  **Preprocess**: Segment tissue from background (Otsu's method).
3.  **Patch Extraction**: Extract 256x256 tiles from tissue regions.
4.  **Dense Inference**: Run ResNet50 on *all* tissue tiles (>50,000).
5.  **Heatmap Generation**: Create the "Gold Standard" high-resolution probability map.

**Part 2: The HASHI Pipeline (Proposed)**
1.  **Sparse Sampling**: Sample only ~5% of tiles using HASHI strategies.
2.  **Reconstruction**: Upsample the sparse predictions to recover the dense map.

**Part 3: Evaluation**
1.  **Ground Truth**: Generate pixel-level tumor masks from XML annotations.
2.  **Comparison**: Compare Dense vs. HASHI performance.

---

## 1. Setup Environment

In [ ]:
!apt-get install -y openslide-tools
!pip install openslide-python wsinfer-zoo timm opencv-python matplotlib scikit-image tqdm shapely pandas

import os
import sys
sys.path.append(os.getcwd())

# Import our local modules
from WSI_load import WSIHandler
from extract_tissue_tiles import extract_tiles_direct
from generate_heatmap import generate_heatmap
from annotation_parser import AnnotationParser
from hashi import HASHISampler

print("Environment Ready.")

## 2. Configuration

In [ ]:
# --- USER INPUT ---
WSI_PATH = "/content/input.tif"
XML_PATH = "/content/annotation.xml"
OUTPUT_ROOT = "/content/output"
# ------------------

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f"Slide: {WSI_PATH}")

--- 
# Part 1: Standard Dense Pipeline
Goal: Generate the high-resolution classification heatmap by classifying EVERY tissue tile.

### Step 1.1: Preprocessing & Tile Extraction
We identify tissue regions and extract all 256x256 patches.

In [ ]:
tiles_dir = os.path.join(OUTPUT_ROOT, "tiles")
csv_path = extract_tiles_direct(WSI_PATH, tiles_dir, patch_size=256, stride=256)
print(f"Tiles extracted to {tiles_dir}")
print(f"Metadata CSV at {csv_path}")

### Step 1.2: Dense Inference (Simulated)
Normally, we would run `wsinfer` on the extracted folder.  
For this notebook, we can run inference within Python using our model utility to populate the CSV with probabilities.

In [ ]:
import pandas as pd
import torch
from model_utils import load_model_with_jit_weights
from torchvision import transforms
from PIL import Image

# Load Model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model_with_jit_weights()
model.to(device)
model.eval()

preprocess = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
])

# Read Extracted CSV
df = pd.read_csv(csv_path)
print(f"Running inference on {len(df)} tiles... (This may take time)")

probs = []
# Batch processing could be added here for speed
for idx, row in df.iterrows():
    img_path = row['file_path']
    img = Image.open(img_path).convert('RGB')
    input_tensor = preprocess(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(input_tensor)
        p = torch.softmax(logits, dim=1)[0, 1].item()
        probs.append(p)

df['prob_Metastasis'] = probs
inference_csv = os.path.join(OUTPUT_ROOT, "dense_inference.csv")
df.to_csv(inference_csv, index=False)
print(f"Inference complete. Results saved to {inference_csv}")

### Step 1.3: Generate Dense Heatmap
Reconstruct the spatial heatmap from the inference CSV.

In [ ]:
heatmap_path = os.path.join(OUTPUT_ROOT, "dense_heatmap.png")
generate_heatmap(inference_csv, WSI_PATH, heatmap_path, downsample_factor=64)

import matplotlib.pyplot as plt
img = plt.imread(heatmap_path)
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.title("Dense Heatmap (Baseline)")
plt.axis('off')
plt.show()

--- 
# Part 2: HASHI Pipeline (Proposed)
Goal: Achieve the same result with <10% of the compute.

In [ ]:
hashi_out = os.path.join(OUTPUT_ROOT, "hashi_results")
sampler = HASHISampler(WSI_PATH, tissue_mask_path=None, device=device)

# Run HASHI Loop
sampler.initial_sampling(200)
sampler.interpolate_map()

for i in range(15):
    sampler.compute_gradients()
    if sampler.adaptive_sampling(50) == 0: break
    sampler.interpolate_map()

sampler.save_state(hashi_out, iteration=99)

# Show Result
plt.figure(figsize=(10, 10))
plt.imshow(sampler.prob_map, cmap='jet', vmin=0, vmax=1)
plt.title(f"HASHI Prediction ({len(sampler.sampled_coords)} samples)")
plt.axis('off')
plt.show()

## Comparison
Comparing standard Dense Sampling vs. Adaptive HASHI.

In [ ]:
print(f"Dense Samples: {len(df)}")
print(f"HASHI Samples: {len(sampler.sampled_coords)}")
print(f"Reduction: {len(sampler.sampled_coords)/len(df):.1%} of original compute used.")